In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('../datasets/smolensk_dataset_shared.csv',sep=';')

In [ ]:
print("Форма таблицы:", df.shape)

In [ ]:
df_dropped = df.iloc[:, 4:]

In [ ]:
df_dropped

In [ ]:
df_dropped.to_csv('smolensk.csv', index=False)

In [ ]:
DATA_CSV = '../datasets/smolensk_clean_guldar.csv'
WEIGHTS_CSV = '../datasets/weights_transport_guldar.csv'
CSV_PATH = 'smolensk.csv'

In [ ]:
COLUMN_RENAME = [
    # Программа 1: формирование комфортной городской среды — дворовые территории
    'бюджет_дворы_pct',
    'дворы_благоустроено_ед',
    'удовлетворенность_средой_дворы_pct',

    # Программа 2: дорожная деятельность и транспорт — источник A
    'бюджет_трансп_A_pct',
    'дороги_отремонт_км_A',
    'дороги_норматив_pct_A',
    'пассажиропоток_тыс_A',
    'рейсы_расписание_pct_A',
    'скорость_магистрали_A_кмч',
    'дтп_10тыс_A',
    'срок_устранения_деф_сут_A',
    'переходы_регулируем_ед_A',

    # Программа 3: адресная социальная поддержка
    'бюджет_соцподдержка_pct',
    'мероприятия_завершено_ед',
    'получатели_адрподдержки_чел',

    # Программа 4: благоустройство общественных территорий
    'бюджет_обществ_территории_pct',
    'территории_благоустроено_ед',
    'удовлетворенность_средой_терр_pct',

    # Программа 5: дорожная деятельность и транспорт — источник B
    # (те же названия показателей, что в программе 2, но другой охват/источник —
    # значения и диапазоны заметно отличаются, это не повтор)
    'бюджет_трансп_B_pct',
    'дороги_отремонт_км_B',
    'дороги_норматив_pct_B',
    'пассажиропоток_тыс_B',
    'рейсы_расписание_pct_B',
    'скорость_магистрали_B_кмч',
    'дтп_10тыс_B',
    'срок_устранения_деф_сут_B',
    'переходы_регулируем_ед_B',

    # Программа 6: дорожная деятельность (узкий срез) — источник C
    'бюджет_дороги_C_pct',
    'дороги_отремонт_км_C',
    'дороги_норматив_pct_C',
    'срок_устранения_деф_сут_C',
]


def clean_csv(csv_path, output_csv):
    # Файлы, выгруженные из Excel с русской локалью, часто используют ";" как
    # разделитель столбцов и "," как десятичный разделитель (вместо "," и "."
    # соответственно) — из-за этого pandas может неверно определить количество
    # столбцов или упасть с ParserError. Перебираем комбинации, пока не найдём
    # рабочую (по совпадению числа столбцов с ожидаемым).
    encodings_to_try = ['utf-8', 'cp1251', 'utf-8-sig', 'latin1']
    seps_to_try = [',', ';', '\t']

    raw_df = None
    used_encoding = None
    used_sep = None
    for enc in encodings_to_try:
        for sep in seps_to_try:
            try:
                candidate = pd.read_csv(csv_path, header=0, encoding=enc, sep=sep)
            except (UnicodeDecodeError, pd.errors.ParserError):
                continue
            if candidate.shape[1] == len(COLUMN_RENAME):
                raw_df = candidate
                used_encoding = enc
                used_sep = sep
                break
        if raw_df is not None:
            break

    if raw_df is None:
        raise ValueError(
            "Не удалось подобрать кодировку/разделитель так, чтобы получить "
            f"{len(COLUMN_RENAME)} столбцов. Проверьте файл вручную (например, "
            "откройте первые строки как обычный текст)."
        )

    print(f"Файл прочитан: кодировка={used_encoding}, разделитель={used_sep!r}")

    raw_df.columns = COLUMN_RENAME

    # приводим всё к числам: сначала меняем десятичную запятую на точку
    # (если разделитель столбцов ";", то запятая внутри чисел — это почти
    # всегда десятичный разделитель, а не часть текста).
    # Проверяем через is_numeric_dtype, а не "dtype == object" — в новых
    # версиях pandas строковые колонки могут иметь dtype 'str'/'string',
    # а не classic 'object', и сравнение с object их не находит.
    for col in raw_df.columns:
        if not pd.api.types.is_numeric_dtype(raw_df[col]):
            raw_df[col] = raw_df[col].astype(str).str.replace(',', '.', regex=False)
        raw_df[col] = pd.to_numeric(raw_df[col], errors='coerce')

    raw_df.to_csv(output_csv, index=False)  # сохраняем результат уже в UTF-8, с "," как разделителем
    return raw_df


CSV_PATH = 'smolensk.csv'
OUTPUT_CSV = '../datasets/smolensk_clean_guldar.csv'
df = clean_csv(CSV_PATH, OUTPUT_CSV)
print(f"Готово: {df.shape[0]} строк, {df.shape[1]} столбцов")
df.head()

In [ ]:
def read_csv_robust(path, expected_min_cols=2):
    """
    Пробует разные кодировки и разделители, пока не получит осмысленную
    таблицу (больше 1 столбца) — полезно для файлов, выгруженных из Excel
    с русской локалью (';' как разделитель, ',' как десятичный знак).
    """
    encodings_to_try = ['utf-8', 'cp1251', 'utf-8-sig', 'latin1']
    seps_to_try = [',', ';', '\t']

    for enc in encodings_to_try:
        for sep in seps_to_try:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep)
            except (UnicodeDecodeError, pd.errors.ParserError):
                continue
            if df.shape[1] >= expected_min_cols:
                print(f"[{path}] прочитан: кодировка={enc}, разделитель={sep!r}, "
                      f"форма={df.shape}")
                return df

    raise ValueError(f"Не удалось прочитать {path} — проверьте кодировку/разделитель вручную")


def fix_decimal_commas(df):
    """Меняет десятичную запятую на точку и приводит колонки к числам,
    где это возможно (нечисловые остаются как есть)."""
    for col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            converted = pd.to_numeric(
                df[col].astype(str).str.replace(',', '.', regex=False),
                errors='coerce'
            )
            # применяем замену только если она не превратила всё в NaN
            # (то есть колонка действительно была числовой с запятой)
            if converted.notna().sum() > 0:
                df[col] = converted
    return df

In [ ]:
class LinearConvolutionIndex:
    """
    Строит составной (синтетический) целевой показатель как взвешенную
    линейную свёртку нескольких признаков, приведённых к одному масштабу:

        target = Σ weight_i * direction_i * normalize(x_i)
    """

    def __init__(self, weights, directions=None, normalization='minmax',
                 normalize_weights=True):
        self.feature_names = list(weights.keys())
        self.raw_weights = np.array([weights[f] for f in self.feature_names], dtype=float)

        if directions is None:
            directions = {f: 1 for f in self.feature_names}
        missing_dir = set(self.feature_names) - set(directions.keys())
        if missing_dir:
            raise ValueError(f"Не заданы направления влияния для признаков: {missing_dir}")
        self.directions = np.array([directions[f] for f in self.feature_names], dtype=float)

        self.normalization = normalization
        self.normalize_weights = normalize_weights

        if normalize_weights:
            self.weights = self.raw_weights / self.raw_weights.sum()
        else:
            self.weights = self.raw_weights

        self._min = None
        self._max = None
        self._mean = None
        self._std = None

    def fit(self, df):
        missing = set(self.feature_names) - set(df.columns)
        if missing:
            raise ValueError(f"В данных нет колонок: {missing}")

        X = df[self.feature_names]

        if self.normalization == 'minmax':
            self._min = X.min()
            self._max = X.max()
        elif self.normalization == 'zscore':
            self._mean = X.mean()
            self._std = X.std().replace(0, 1)
        elif self.normalization != 'none':
            raise ValueError("normalization должно быть 'minmax', 'zscore' или 'none'")

        return self

    def normalize(self, df):
        X = df[self.feature_names].astype(float)

        if self.normalization == 'minmax':
            rng = (self._max - self._min).replace(0, 1)
            return (X - self._min) / rng
        elif self.normalization == 'zscore':
            return (X - self._mean) / self._std
        else:
            return X

    def transform(self, df):
        X_norm = self.normalize(df)
        signed = X_norm * self.directions
        index = signed @ self.weights
        return pd.Series(index, index=df.index, name='индекс')

    def fit_transform(self, df):
        return self.fit(df).transform(df)

    def contributions(self, df):
        X_norm = self.normalize(df)
        contrib = X_norm * self.directions * self.weights
        contrib['индекс_итог'] = contrib.sum(axis=1)
        return contrib

    def weights_table(self):
        return pd.DataFrame({
            'Признак': self.feature_names,
            'Экспертный вес (исходный)': self.raw_weights,
            'Вес (нормированный)': self.weights,
            'Направление влияния': np.where(self.directions > 0, '+', '-'),
        })

In [ ]:
data_df = read_csv_robust(DATA_CSV)
data_df = fix_decimal_commas(data_df)

weights_df = read_csv_robust(WEIGHTS_CSV)
weights_df = fix_decimal_commas(weights_df)

weights = dict(zip(weights_df['feature'], weights_df['weight']))
if 'direction' in weights_df.columns:
    directions = dict(zip(weights_df['feature'], weights_df['direction']))
else:
    directions = {f: 1 for f in weights}

print(f"\nПризнаков в данных: {data_df.shape[1]}, признаков с весами: {len(weights)}")
missing_in_data = set(weights.keys()) - set(data_df.columns)
if missing_in_data:
    print("ВНИМАНИЕ: этих признаков из файла весов нет в данных:", missing_in_data)

In [ ]:
model = LinearConvolutionIndex(weights, directions, normalization='minmax')
model.fit(data_df)

normalized = model.normalize(data_df)
print("\n=== Нормализованные признаки (первые 10 строк) ===")
display(normalized.head(10))

index = model.transform(data_df)
result = data_df.copy()
result['индекс'] = index
print("\n=== Целевой показатель (первые 10 строк) ===")
display(result[['индекс']].head(10))

print("\n=== Таблица весов ===")
display(model.weights_table())

In [ ]:
index = result['индекс']

In [ ]:
print("Минимум:          %.4f" % index.min())
print("Максимум:         %.4f" % index.max())
print("Среднее:          %.4f" % index.mean())
print("Медиана:          %.4f" % index.median())
print("Станд. отклонение: %.4f" % index.std())

print("\nПолная сводка (describe):")
print(index.describe().to_string())

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(index, bins=15, edgecolor='black', alpha=0.7)
plt.axvline(index.mean(), color='red', linestyle='--', label=f'Среднее = {index.mean():.3f}')
plt.axvline(index.mean() - index.std(), color='orange', linestyle=':', label=f'-1 std = {index.mean()-index.std():.3f}')
plt.axvline(index.mean() + index.std(), color='orange', linestyle=':', label=f'+1 std = {index.mean()+index.std():.3f}')
plt.xlabel('Значение индекса')
plt.ylabel('Количество наблюдений')
plt.title('Распределение целевого показателя (индекс транспортной доступности)')
plt.legend()
plt.tight_layout()
plt.show()